# Estudio Yanero: entrena tu voz en Google Colab

Para cuando no tienes GPU NVIDIA en tu PC. Colab te presta una gratis.

**Antes de empezar:** en tu Google Drive crea la carpeta `EstudioYanero/grabaciones` y sube ahí tus grabaciones cantando (audio o video, 10–30 min en total).

**Luego:** *Entorno de ejecución → Cambiar tipo de entorno → GPU T4*, y *Entorno de ejecución → Ejecutar todas*.

Al final la voz entrenada queda en `EstudioYanero/voces/` de tu Drive y se descarga sola.
Si Colab se desconecta a mitad del entrenamiento (en la versión gratis pasa tras unas horas), vuelve a ejecutar todo: **continúa donde se quedó**.


In [ ]:
#@title 1. Comprobar GPU y conectar Google Drive
import subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU T4 y vuelve a ejecutar.")
print("GPU:", gpu.stdout.strip())
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#@title 2. Ajustes
NOMBRE_VOZ = "mi-voz"  #@param {type:"string"}
SEPARAR_MUSICA = True  #@param {type:"boolean"}
CARPETA = "/content/drive/MyDrive/EstudioYanero"  #@param {type:"string"}

import pathlib
grab = pathlib.Path(CARPETA) / "grabaciones"
exts = {".wav",".mp3",".flac",".m4a",".aac",".ogg",".opus",".mp4",".mov",".mkv",".webm",".avi",".m4v"}
archivos = [p for p in grab.rglob("*") if p.suffix.lower() in exts] if grab.exists() else []
print(f"{len(archivos)} grabaciones encontradas en {grab}")
assert archivos, "Sube tus grabaciones a la carpeta EstudioYanero/grabaciones de tu Drive."


In [ ]:
#@title 3. Instalar (unos 5–10 minutos)
REPO = "https://github.com/yanerox69/estudio-yanero.git"
%cd /content
!test -d /content/estudio-yanero/.git || git clone -q --depth 1 "{REPO}" /content/estudio-yanero
!git -C /content/estudio-yanero pull -q --ff-only || true
!test -d /content/estudio-yanero/Applio || (git clone -q https://github.com/IAHispano/Applio.git /content/estudio-yanero/Applio && git -C /content/estudio-yanero/Applio checkout -q 94f5d69)
!pip install -q -r /content/estudio-yanero/Applio/requirements-engine.txt --extra-index-url https://download.pytorch.org/whl/cu128
!pip install -q -r /content/estudio-yanero/requirements.txt
%cd /content/estudio-yanero/Applio
!python core.py prerequisites --pretraineds-hifigan --models --exe
%cd /content/estudio-yanero
print("Instalación lista")


In [ ]:
#@title 4. Entrenar (1–4 horas). Si se desconecta, vuelve a ejecutar todo
import os
os.environ["ESTUDIO_APPLIO"] = "/content/estudio-yanero/Applio"
datos = f"{CARPETA}/datos"
salida = f"{CARPETA}/voces/{NOMBRE_VOZ}_voz_entrenada.zip"
os.makedirs(f"{CARPETA}/voces", exist_ok=True)
sep = "--separar" if SEPARAR_MUSICA else ""
!cd /content/estudio-yanero && python -m estudio.entrenar_cli "{grab}" --nombre "{NOMBRE_VOZ}" --datos "{datos}" --salida "{salida}" {sep}

In [ ]:
#@title 5. Descargar la voz entrenada
import os
from google.colab import files
if os.path.exists(salida):
    print("Guardada también en tu Drive:", salida)
    files.download(salida)
else:
    print("Aún no hay voz entrenada. Revisa el mensaje del paso 4.")

**Para usarla:** en Estudio Yanero de tu PC, sección *Mis voces entrenadas → Importar*, sube el zip y luego usa *Cantar con mi voz*.
El zip trae `.pth` + `.index` estándar de RVC v2, así que también funciona en Applio o en cualquier programa compatible.
